There are times when we'll require to manually access the event loop and it's methods / features...

Maybe to override default event loop behaviour or something else... These scenarios will be rare, but we must still know how to do this.

### 1. Creating an event loop manually

In [ ]:
import asyncio

async def main():
    await asyncio.sleep(1)

loop = asyncio.new_event_loop()

try:
    loop.run_until_complete(main()) # runs the function until it's complete.. Once coroutine is done, loop closes with it.
finally:
    loop.close() # clean up resources used by the loop, you can add custom cleanup logic here..

# there's also loop.run_forever(), which runs loop forever, until stop() is called explicitly.

### 2. Accessing event loop

In [ ]:
import asyncio
from util import delay

def call_later():
    print("I'm being called in the future!")

async def main():
    loop = asyncio.get_running_loop() # get the currently running event loop
    loop.call_soon(call_later) # call_soon = call on the next iteration of event loop
    await delay(1)

asyncio.run(main())

# NOTE: loop.call_soon()` schedules a synchronous callback for the next event loop iteration. It runs the function immediately (non-blocking), not as a coroutine.

# If `call_later` were `async`, you'd need `loop.create_task()` instead to actually await it. `call_soon()` just invokes the callable directly — it doesn't know how to handle coroutines.

### 3. Using debug mode

you may need debug mode to understand if your coroutines are silently passing some exception, or if a coroutine is taking too long on the CPU etc etc...

tbh I can't appreciate debug mode yet, and I probably won't until i actually have to use it.

3 ways to turn on debug mode - 

```python
asyncio.run(coroutine(), debug=True)
```

```sh
python -x dev program.py
```

or set the `PYTHONASYNCIODEBUG` env variable to 1.

In [ ]:
import asyncio
from util import async_timed, delay

@async_timed()
async def cpu_bound_work() -> int:
    counter = 0
    for i in range(100000000):
        counter = counter + 1
    return counter

async def main() -> None:
    loop = asyncio.get_running_loop()
    loop.slow_callback_duration = 0.1 # 100ms is default

    task_one = asyncio.create_task(cpu_bound_work())
    task_two = delay(1)
    task_three = delay(1)
    await task_one
    await task_two
    await task_three

asyncio.run(main(), debug=True)

# NOTE: the debug mode will log a warning whenever a given coroutine has continuous control between 2 yields to the event loop for a duration longer than 'slow_callback_duration'.. This is a means for understanding which points in your application take too long between 2 awaits.. just think of it as one uninterrupted slice of CPU time... If a given callback is using the cpu too long in one sitting, chances are - 

# 1. you're running cpu-intensive code there. Your best bet is to use a process pool here for concurrency.
# 2. you're running blocking api calls, shift to a framework that uses non-blocking sockets, or using the thread pool executor here...
# 3. you missed writing an await in the code in that coroutine, calling a synchronous function instead.

# 'slow_callback_duration' is an attribute of the event loop, and is set to 100ms on default.. You can increase / decrease this as shown in code